# 📐 Reducción de Dimensionalidad con PCA & Clasificación Litofacies 3D
## Módulo 4 · Compresión de Well Logs & Plotly 3D · Capacitación SLB

### Objetivos de la sesión:
1. Comprimir 4 curvas de registros geofísicos correlacionados (`GR`, `ILD`, `RHOB`, `NPHI`) usando PCA.
2. Evaluar la varianza explicada acumulada con un Scree Plot para retener >90% de información.
3. Clasificar facies con K-Means y renderizar un volumen tridimensional interactivo con **Plotly 3D**.

## ¿Qué librerías utilizaremos para PCA y renderizado 3D en Plotly?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

# 1. Carga y Estandarización de Registros Geofísicos

## ¿Cómo descargamos `registros_pozo.csv` desde GitHub y cargamos las curvas de subsuelo?

In [ ]:
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_4_no_supervisado/data/registros_pozo.csv -O registros_pozo.csv

df_logs = pd.read_csv('registros_pozo.csv')
df_logs.head()

## ¿Cómo estandarizamos las 4 curvas geofísicas (`GR_API`, `ILD_ohm_m`, `RHOB_g_cc`, `NPHI_v_v`)?

In [ ]:
log_features = ['GR_API', 'ILD_ohm_m', 'RHOB_g_cc', 'NPHI_v_v']

# TU CÓDIGO AQUÍ: Escala las variables seleccionadas con StandardScaler
scaler = StandardScaler()
X_logs_scaled = scaler.fit_transform(df_logs[log_features])

# 2. Reducción de Dimensionalidad con PCA

## ¿Cómo entrenamos PCA de 2 componentes y visualizamos la varianza explicada acumulada (Scree Plot)?

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_logs_scaled)

print("Varianza explicada por cada componente:", pca.explained_variance_ratio_.round(4))
print("Varianza acumulada total:", np.sum(pca.explained_variance_ratio_).round(4))

# TU CÓDIGO AQUÍ: Asigna PC1 y PC2 al DataFrame df_logs
df_logs['PC1'] = X_pca[:, 0]
df_logs['PC2'] = X_pca[:, 1]
df_logs.head()

# 3. Clasificación de Litofacies y Renderizado 3D

## ¿Cómo aplicamos K-Means sobre el espacio reducido (PC1 y PC2) para identificar 3 Litofacies?

In [ ]:
# TU CÓDIGO AQUÍ: Entrena KMeans con n_clusters=3 sobre X_pca y guarda la columna 'Facies'
kmeans_facies = KMeans(n_clusters=3, random_state=42)
df_logs['Facies'] = kmeans_facies.fit_predict(X_pca)

df_logs.groupby('Facies')[log_features].mean().round(2)

## ¿Cómo visualizamos las litofacies clasificadas en un espacio 3D interactivo con Plotly?

In [ ]:
# TU CÓDIGO AQUÍ: Renderiza un scatter_3d de Plotly usando PC1, PC2 y Profundidad_m coloreado por Facies
fig = px.scatter_3d(
    df_logs, 
    x='PC1', 
    y='PC2', 
    z='Profundidad_m', 
    color='Facies',
    title='Clasificación 3D de Litofacies mediante PCA y K-Means'
)
fig.update_scenes(zaxis_autorange="reversed") # Invertir profundidad en eje Z
fig.show()

> **🔍 Observación:** El modelo redujo exitosamente la redundancia de las 4 curvas geofísicas en 2 Componentes Principales con >90% de varianza, y Plotly 3D nos permite rotar el volumen del pozo para inspeccionar visualmente los bloques de facies a lo largo de la profundidad.